# System experiment analysis

Analysis of aggregated system-experiment results.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "metacognitive_experiments").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Run this notebook from inside the repository")
    ROOT = ROOT.parent

# One or more aggregated result files to compare.
RESULTS = [
    ROOT / "results" / "system_results_pogema_pogema.csv",
]
SYSTEM_LABELS = {
    "lacam_lns": "S1+S2 default - early stopping",
    "lacam_naive": "S1 default - early stopping ",
    "learnable_fixed": "learnable-fixed-params",
    "learnable": "learnable",
    "layered_v1": "learnable-fixed-params",  # Legacy result files.
    "layered_variable_neighbourhood_v1": "learnable",
    "naive_budget" : "naive budget - early stopping" 
}
AUDIT_FEATURES = [
    "num_agents", "num_obstacles", "agent_density", "obstacle_density",
    "avg_shortest_path_distance", "min_shortest_path_distance",
    "max_shortest_path_distance", "cells_at_sp_ratio",
    "num_total_cells", "num_free_cells", "lower_bound_soc",
    "sp_collision_count", "sp_vertex_collision_count",
    "sp_edge_collision_count", "lacam_initial_sod", "initial_sodlb",
    "lacam_fraction_delayed_agents", "lacam_avg_delay",
    "lacam_delay_90th_percentile",
]
OUTPUT = RESULTS[0].parent
for path in RESULTS:
    if not path.exists():
        raise FileNotFoundError(f"Aggregate the experiment first: {path}")

columns = set().union(*(pd.read_csv(path, nrows=0).columns for path in RESULTS))
parameters = sorted(name for name in columns if name.startswith("parameter_"))
SYSTEM_COLUMNS = ("system", "metacognitive_module", "policy")
required = {
    "parameter_budget", "repetition", "task", "position",
    "instance_id", "solved", "budget_after", "sequence_elapsed_seconds",
    "s1_wall_seconds", "s2_wall_seconds", "s1_requested_seconds",
    "s2_requested_seconds", "s2_neighborhood_size", "s1_status", "s2_status",
    "initial_sodlb", "final_sodlb", "sodlb_improvement",
    *AUDIT_FEATURES,
}
# Full traces stay in task checkpoints; only compact decision metadata is loaded here.
usecols = required | set(parameters) | set(SYSTEM_COLUMNS) | ({"s1_info", "s2_info"} & columns)
frames = []
for path in RESULTS:
    available = set(pd.read_csv(path, nrows=0).columns)
    source_system_columns = [name for name in SYSTEM_COLUMNS if name in available]
    if len(source_system_columns) != 1:
        raise KeyError(f"{path} must contain exactly one system column; found {source_system_columns}")
    if missing := sorted(required - available):
        raise KeyError(f"{path} is missing {missing}")
    frame = pd.read_csv(path, usecols=sorted(usecols & available))
    frame = frame.rename(columns={source_system_columns[0]: "system"})
    frame["system"] = frame["system"].replace(SYSTEM_LABELS)
    frame["_result_file"] = str(path)
    if len(RESULTS) > 1:
        frame["system"] = path.parent.name + " · " + frame["system"]
    frames.append(frame)
data = pd.concat(frames, ignore_index=True).sort_values(["system", "task", "position"])
systems = data["system"].drop_duplicates().tolist()
system_colors = dict(zip(systems, plt.get_cmap("tab10").colors))

sequence_count = data.groupby(["system", "task"]).ngroups
print(f"{len(data):,} instance decisions across {sequence_count} sequences")
display(data.groupby("system").agg(
    sequences=("task", "nunique"),
    budgets=("parameter_budget", lambda values: sorted(values.unique())),
))

## Sequence-level results

In [ ]:
sequence_keys = ["system", "parameter_budget", "repetition", "task"]
sequences = data.groupby(sequence_keys, as_index=False).agg(
    instances=("instance_id", "size"),
    solved_instances=("solved", "sum"),
    solve_rate=("solved", "mean"),
    remaining_seconds=("budget_after", "last"),
    elapsed_seconds=("sequence_elapsed_seconds", "last"),
    lacam_seconds=("s1_wall_seconds", "sum"),
    lns_seconds=("s2_wall_seconds", "sum"),
    lns_runs=("s2_wall_seconds", lambda values: values.gt(0).sum()),
    mean_initial_sodlb=("initial_sodlb", "mean"),
    mean_final_sodlb=("final_sodlb", "mean"),
    mean_improvement=("sodlb_improvement", "mean"),
)
sequences["charged_seconds"] = sequences.parameter_budget - sequences.remaining_seconds

by_run = sequences.groupby(["system", "parameter_budget"]).agg(
    instances=("instances", "mean"),
    solved_instances=("solved_instances", "mean"),
    solve_rate=("solve_rate", "mean"),
    solve_rate_sd=("solve_rate", "std"),
    elapsed_seconds=("elapsed_seconds", "mean"),
    charged_seconds=("charged_seconds", "mean"),
    remaining_seconds=("remaining_seconds", "mean"),
    lacam_seconds=("lacam_seconds", "mean"),
    lns_seconds=("lns_seconds", "mean"),
    lns_runs=("lns_runs", "mean"),
    mean_initial_sodlb=("mean_initial_sodlb", "mean"),
    mean_final_sodlb=("mean_final_sodlb", "mean"),
    final_sodlb_sd=("mean_final_sodlb", "std"),
    mean_improvement=("mean_improvement", "mean"),
    improvement_sd=("mean_improvement", "std"),
)
display(by_run.round(4))

# Within each matched budget and repetition, retain instances solved by every system.
# Comparisons are paired across systems without requiring success in every repetition.
paired_parts = []
coverage_rows = []
for (budget, repetition), rows in data.groupby(["parameter_budget", "repetition"]):
    solved_sets = [
        set(run.loc[run.solved, "instance_id"])
        for _, run in rows.groupby(["system", "task"])
    ]
    common = set.intersection(*solved_sets) if solved_sets else set()
    universe = set(rows.instance_id)
    coverage_rows.append({
        "parameter_budget": budget,
        "repetition": repetition,
        "common_instances": len(common),
        "total_instances": len(universe),
        "coverage": len(common) / len(universe) if universe else np.nan,
    })
    if common:
        paired_parts.append(rows[rows.instance_id.isin(common)])

coverage = pd.DataFrame(coverage_rows).groupby("parameter_budget", as_index=False).agg(
    common_instances=("common_instances", "mean"),
    common_instances_min=("common_instances", "min"),
    total_instances=("total_instances", "mean"),
    coverage=("coverage", "mean"),
    coverage_sd=("coverage", "std"),
).sort_values("parameter_budget")
paired_data = pd.concat(paired_parts, ignore_index=True) if paired_parts else data.iloc[:0]
paired_sequences = paired_data.groupby(sequence_keys, as_index=False).agg(
    mean_final_sodlb=("final_sodlb", "mean"),
    mean_improvement=("sodlb_improvement", "mean"),
)
paired_by_run = paired_sequences.groupby(["system", "parameter_budget"]).agg(
    mean_final_sodlb=("mean_final_sodlb", "mean"),
    final_sodlb_sd=("mean_final_sodlb", "std"),
    mean_improvement=("mean_improvement", "mean"),
    improvement_sd=("mean_improvement", "std"),
).reset_index()
paired_by_run["mean_negative_final_sodlb"] = -paired_by_run.mean_final_sodlb

display(coverage.round(4))


def plot_band(axis, rows, mean, spread, color, label, bounds=None):
    rows = rows.sort_values("parameter_budget")
    x = rows.parameter_budget.to_numpy()
    y = rows[mean].to_numpy()
    sd = rows[spread].fillna(0).to_numpy()
    lower, upper = y - sd, y + sd
    if bounds is not None:
        lower, upper = np.clip(lower, *bounds), np.clip(upper, *bounds)
    axis.plot(x, y, marker="o", color=color, label=label)
    axis.fill_between(x, lower, upper, color=color, alpha=0.15)


fig, axis = plt.subplots(figsize=(7, 4.5))
overview_rows = by_run.reset_index()
overview_rows = overview_rows[
    ~overview_rows.system.str.endswith(SYSTEM_LABELS["naive_budget"])
]
for system, rows in overview_rows.groupby("system"):
    plot_band(axis, rows, "solve_rate", "solve_rate_sd", system_colors[system], system, (0, 1))
axis.set(title="Instances solved", xlabel="System budget (s)", ylabel="Mean fraction solved", ylim=(0, 1.02))
axis.grid(alpha=0.25)
axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT / "system_overview.png", dpi=180)
plt.show()


In [ ]:
# Compare the two learned systems with naive_budget on the same solved instances.
paired_system_suffixes = ("learnable-fixed-params", "learnable", "naive budget - early stopping")
three_way_rows = []
three_way_coverage_rows = []

for (result_file, budget, repetition), rows in data.groupby(
    ["_result_file", "parameter_budget", "repetition"], sort=False
):
    matched = {}
    for suffix in paired_system_suffixes:
        names = rows.loc[rows.system.str.endswith(suffix), "system"].unique()
        if len(names) == 1:
            matched[suffix] = rows.loc[rows.system.eq(names[0])]
    if len(matched) != len(paired_system_suffixes):
        continue

    solved_sets = [set(run.loc[run.solved, "instance_id"]) for run in matched.values()]
    common_instances = set.intersection(*solved_sets)
    total_instances = len(set.union(*(set(run.instance_id) for run in matched.values())))
    three_way_coverage_rows.append({
        "_result_file": result_file,
        "parameter_budget": budget,
        "repetition": repetition,
        "common_instances": len(common_instances),
        "total_instances": total_instances,
        "coverage": len(common_instances) / total_instances if total_instances else np.nan,
    })
    if not common_instances:
        continue

    for run in matched.values():
        common = run.loc[run.instance_id.isin(common_instances)]
        three_way_rows.append({
            "_result_file": result_file,
            "system": common.system.iloc[0],
            "parameter_budget": budget,
            "repetition": repetition,
            "common_instances": len(common_instances),
            "mean_final_sodlb": common.final_sodlb.mean(),
            "mean_improvement": common.sodlb_improvement.mean(),
        })

three_way_sequences = pd.DataFrame(three_way_rows)
if three_way_sequences.empty:
    print("No result-file/budget/repetition group contains all three requested systems with a common solved instance.")
else:
    three_way_summary = three_way_sequences.groupby(
        ["system", "parameter_budget"], as_index=False
    ).agg(
        mean_final_sodlb=("mean_final_sodlb", "mean"),
        final_sodlb_sd=("mean_final_sodlb", "std"),
        mean_improvement=("mean_improvement", "mean"),
        improvement_sd=("mean_improvement", "std"),
    )
    three_way_summary["mean_negative_final_sodlb"] = -three_way_summary.mean_final_sodlb

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for system, system_rows in three_way_summary.groupby("system"):
        color = system_colors[system]
        plot_band(axes[0], system_rows, "mean_negative_final_sodlb", "final_sodlb_sd", color, system)
        plot_band(axes[1], system_rows, "mean_improvement", "improvement_sd", color, system)

    axes[0].set(
        title="Quality on instances solved by all three",
        xlabel="System budget (s)",
        ylabel="Mean −(final SoD/LB) (higher is better)",
    )
    axes[1].set(
        title="Quality improvement on instances solved by all three",
        xlabel="System budget (s)",
        ylabel="Mean SoD/LB improvement (higher is better)",
    )
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT / "system_learned_vs_naive_paired.png", dpi=180)
    plt.show()


## Time allocation by system

In [ ]:
for system in systems:
    fig, axis = plt.subplots(figsize=(15, 4))
    rows = by_run.loc[system].reset_index()
    x = np.arange(len(rows))
    other = (rows.charged_seconds - rows.lacam_seconds - rows.lns_seconds).clip(lower=0)
    axis.bar(x, rows.lacam_seconds, label="LaCAM")
    axis.bar(x, rows.lns_seconds, bottom=rows.lacam_seconds, label="LNS")
    axis.bar(x, other, bottom=rows.lacam_seconds + rows.lns_seconds, label="Features/decisions")
    axis.plot(x, rows.parameter_budget, "k--", label="System budget")
    overhead_percent = 100 * other / rows.parameter_budget.replace(0, np.nan)
    stack_top = rows.lacam_seconds + rows.lns_seconds + other
    for position, top, percent in zip(x, stack_top, overhead_percent):
        axis.annotate(
            f"{percent:.1f}%",
            (position, top),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=7,
        )
    axis.set(
        title=system, xlabel="System budget (s)", ylabel="Mean measured seconds",
        xticks=x, xticklabels=[f"{value:g}" for value in rows.parameter_budget],
    )
    axis.tick_params(axis="x", labelrotation=90)
    axis.grid(axis="y", alpha=0.25)
    axis.legend()
    fig.tight_layout()
    filename = "system_time_allocation_" + system.lower().replace(" ", "_").replace("+", "plus") + ".png"
    fig.savefig(OUTPUT / filename, dpi=180, bbox_inches="tight")
    plt.show()

## Expected versus realized S2 gain

Projected gain is the S2 model's prediction; realized gain is the observed improvement. Values are averaged across repetitions, and horizon 0 means that S2 was skipped.

In [ ]:
learned = data.system.str.endswith(("learnable_fixed", "learnable"))
decisions = data[learned & data.initial_sodlb.notna()].copy()

def parse_info(value):
    if not isinstance(value, str) or not value.strip():
        return {}
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return {}

info = decisions.s2_info.map(parse_info)
selected = decisions.s2_requested_seconds.gt(0)
decisions["horizon"] = decisions.s2_requested_seconds.where(selected, 0).astype(int)
decisions["neighbourhood"] = np.where(
    selected,
    "n" + decisions.s2_neighborhood_size.fillna(0).astype(int).astype(str),
    "skip",
)
decisions["projected_gain"] = [float(item.get("predicted_improvement", 0.0)) for item in info]
decisions["actual_gain"] = decisions.sodlb_improvement.fillna(0.0)

sequence_gain = decisions.groupby(
    ["system", "parameter_budget", "repetition", "task"], as_index=False
).agg(
    projected_gain=("projected_gain", "sum"),
    actual_gain=("actual_gain", "sum"),
    s2_selections=("horizon", lambda values: values.gt(0).sum()),
)
gain_summary = sequence_gain.groupby(["system", "parameter_budget"]).agg(
    projected_total=("projected_gain", "mean"),
    projected_sd=("projected_gain", "std"),
    actual_total=("actual_gain", "mean"),
    actual_sd=("actual_gain", "std"),
    s2_selections=("s2_selections", "mean"),
).reset_index()

naive = data[
    data.system.str.endswith(SYSTEM_LABELS["naive_budget"])
    & data.initial_sodlb.notna()
].copy()
naive["actual_gain"] = naive.sodlb_improvement.fillna(0.0)
naive_sequence = naive.groupby(
    ["_result_file", "parameter_budget", "repetition", "task"], as_index=False
).agg(actual_gain=("actual_gain", "sum"))
naive_summary = naive_sequence.groupby(
    ["_result_file", "parameter_budget"], as_index=False
).agg(actual_total=("actual_gain", "mean"), actual_sd=("actual_gain", "std"))
system_source = decisions.groupby("system")["_result_file"].first()
display(gain_summary[[
    "system", "parameter_budget", "s2_selections",
    "projected_total", "actual_total",
]].round(4))

fig, axes = plt.subplots(len(gain_summary.system.unique()), 1, figsize=(12, 4 * len(gain_summary.system.unique())), squeeze=False)
for axis, (system, rows) in zip(axes.flat, gain_summary.groupby("system", sort=False)):
    rows = rows.sort_values("parameter_budget")
    x = rows.parameter_budget.to_numpy()
    for mean, spread, label, color in [
        ("projected_total", "projected_sd", "Projected", "tab:orange"),
        ("actual_total", "actual_sd", "Realized", "tab:blue"),
    ]:
        y = rows[mean].to_numpy()
        sd = rows[spread].fillna(0).to_numpy()
        axis.plot(x, y, marker="o", color=color, label=label)
        axis.fill_between(x, y - sd, y + sd, color=color, alpha=0.15)
    baseline = naive_summary[naive_summary._result_file.eq(system_source[system])].sort_values("parameter_budget")
    if not baseline.empty:
        bx = baseline.parameter_budget.to_numpy()
        by = baseline.actual_total.to_numpy()
        bsd = baseline.actual_sd.fillna(0).to_numpy()
        axis.plot(bx, by, marker="o", linestyle="--", color="tab:green", label="Naive budget realized")
        axis.fill_between(bx, by - bsd, by + bsd, color="tab:green", alpha=0.12)
    axis.set(title=system, xlabel="System budget (s)", ylabel="Total SoD/LB gain per sequence")
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT / "s2_expected_vs_realized.png", dpi=180, bbox_inches="tight")
plt.show()

gain_summary["realized_shortfall_pct"] = 100 * (
    gain_summary.projected_total - gain_summary.actual_total
) / gain_summary.projected_total.replace(0, np.nan)
shortfall_table = gain_summary.pivot(
    index="parameter_budget",
    columns="system",
    values="realized_shortfall_pct",
).round(1)
shortfall_table.index.name = "budget (s)"
shortfall_table.columns.name = "realized gain below prediction (%)"
display(shortfall_table)

sequence_counts = decisions[["system", "parameter_budget", "repetition", "task"]].drop_duplicates().groupby(
    ["system", "parameter_budget"]
).size().rename("sequences")
action_summary = decisions.groupby(
    ["system", "parameter_budget", "horizon", "neighbourhood"], as_index=False
).agg(
    selections=("horizon", "size"),
    projected_total=("projected_gain", "sum"),
    actual_total=("actual_gain", "sum"),
)
action_summary = action_summary.join(sequence_counts, on=["system", "parameter_budget"])
action_summary["projected_per_selection"] = action_summary.projected_total / action_summary.selections
action_summary["actual_per_selection"] = action_summary.actual_total / action_summary.selections
action_summary[["selections", "projected_total", "actual_total"]] = action_summary[[
    "selections", "projected_total", "actual_total"
]].div(action_summary.sequences, axis=0)
action_summary = action_summary.drop(columns="sequences")
for system, rows in action_summary.groupby("system", sort=False):
    print(system)
    display(rows.drop(columns="system").round(4).set_index(["parameter_budget", "horizon", "neighbourhood"]))


## System scores

Scores are averaged across repetitions. Final SoD/LB uses solved instances; improvement is averaged over every instance in the sequence.

In [ ]:
sequence_scores = (
    data.groupby(
        ["system", "parameter_budget", "repetition", "task"],
        as_index=False,
    ).agg(
        instances=("instance_id", "size"),
        solved_instances=("solved", "sum"),
        mean_final_sodlb=("final_sodlb", "mean"),
        total_sodlb_improvement=("sodlb_improvement", "sum"),
    )
)
sequence_scores["mean_sodlb_improvement"] = (
    sequence_scores.total_sodlb_improvement / sequence_scores.instances
)
scores = (
    sequence_scores.groupby(["parameter_budget", "system"], as_index=False)
    .agg(
        solved_instances=("solved_instances", "mean"),
        mean_final_sodlb=("mean_final_sodlb", "mean"),
        mean_sodlb_improvement=("mean_sodlb_improvement", "mean"),
    )
)
scores["score"] = [
    (round(solved, 1), round(quality, 4), round(improvement, 4))
    for solved, quality, improvement in zip(
        scores.solved_instances,
        scores.mean_final_sodlb,
        scores.mean_sodlb_improvement,
    )
]
score_table = scores.pivot(
    index="parameter_budget",
    columns="system",
    values="score",
).sort_index(axis=1)
score_table.index.name = "budget (s)"
score_table.columns.name = (
    "system: (solved instances, mean final SoD/LB, "
    "mean SoD/LB improvement per instance)"
)
display(score_table)
